# Pipeline Multiclasse (25% / 50% / 75%) v2 — RandomForest + Regressão Logística + XGBoost, com CV Aninhada

**Versão 2** — volta a usar as 3 condições (25%/50%/75%, multiclasse) e corrige os
pontos levantados na revisão do notebook v1 (CV aninhada 50 vs 75):

| Ponto da revisão | O que era | O que virou |
|---|---|---|
| **1. área vira amostra separada** | uma linha por `(trial, área)` -- o modelo nunca via as 4 áreas juntas do mesmo trial | uma linha por **trial**, com as áreas em **blocos de colunas** (`CA3b_pot_theta_...`, `CA3c_pot_theta_...` etc.) |
| **4. `passo_pct` inconsistente** | `n_bins = round(100/passo_pct)` fazia `30` e `35` (ou `40`/`45`/`50`) colapsarem no mesmo `n_bins` -- grade com duplicatas mascaradas | grade direta em **`N_BINS_GRID`** (`[2, 3, 4, 5, 8]`), sem o arredondamento |
| **6. unidade da métrica** | acurácia calculada por linha (= por área) | resolvido de graça pelo ponto 1 -- já é 1 previsão por trial |
| **7. `arquivo_area` sem condição** | `f"{rato}_{trial}_{area}"`, risco de colisão entre condições | não existe mais -- cada linha já é identificada só por `stem` |
| **8. filtro NOR por nome de arquivo** | `stem.str.contains("nor")` | filtro primário pela coluna `condicao` (`isin({25,50,75})`), com o `.str.contains("nor")` como camada extra de segurança, + `assert` |

Os pontos **2** (correlação calculada com o dataset inteiro) e **3** (não era nested CV)
já tinham sido corrigidos na v1 (CV aninhada) e continuam corrigidos aqui.

Os pontos **5** (N real é 7 ratos, não milhares de linhas) e **9** (pode ser mais
biológico que algorítmico) não são bugs de código -- são limitações a discutir no
relatório, não algo pra "consertar".

> ⚠️ **Atenção ao tamanho do dataset**: colocar as áreas como blocos de colunas
> multiplica o número de atributos por ~4 (uma vez por área) na mesma linha, e agora
> as linhas (trials) ficaram ~4x menos numerosas (antes cada trial virava até 4 linhas,
> uma por área). Com poucos trials e muitas colunas, a seleção de features e a remoção
> de correlacionados ficam ainda mais importantes -- e ainda mais caras computacionalmente.
> `N_BINS_GRID` foi deixado moderado (`[2, 3, 4, 5, 8]`) de propósito; veja o comentário
> na Seção 1 antes de ampliar.

> Este notebook **não foi executado aqui** — não tenho acesso aos seus arquivos de dados.
> Rode do zero no seu ambiente.

## 1. Configuração e Carregamento dos Dados

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.ensemble import RandomForestClassifier  # usado também como selector interno do SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

try:
    import shap
    SHAP_DISPONIVEL = True
except ImportError:
    SHAP_DISPONIVEL = False
    print("Pacote 'shap' não encontrado. Instale com: pip install shap")

try:
    from xgboost import XGBClassifier
    XGBOOST_DISPONIVEL = True
except ImportError:
    XGBOOST_DISPONIVEL = False
    print("Pacote 'xgboost' não encontrado. Instale com: pip install xgboost")

warnings.filterwarnings("ignore")
# Filtro extra para o warning de paralelismo aninhado do sklearn -- a causa real
# já está corrigida (estimadores internos com n_jobs=1), isso é só reforço.
warnings.filterwarnings("ignore", message=".*should be used with.*Parallel.*")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

TOP_N_FEATURES_GRID = [10, 15, 20, 30]

# Caminho do features_all
CAMINHOS_CANDIDATOS = [
    Path("databases/features_all.csv"),
]

# Limite mínimo de duração do vídeo (segundos)
DURACAO_MINIMA_S = 30

FEATURE_BOOLEANA = "pac_sl_alerta"
ESTATISTICAS_POR_JANELA = ["mean", "std", "min", "max", "median"]
FEATURES_POR_AREA = [
    "mean", "std", "rms", "kurtosis", "skewness", "pico_a_pico", "pct_outlier_3s",
    "pot_delta", "pot_theta", "pot_beta", "pot_gamma_lento", "pot_gamma_rapido",
    "theta_gamma_lento_ratio", "theta_gamma_rapido_ratio", "delta_theta_ratio",
    "entropia_espectral", "centroide_hz",
    "pac_theta-gamma_rapido", "pac_theta-gamma_lento", "pac_sl_alerta",
]

# CORREÇÃO DO PONTO 4: grade direta em número de bins, em vez de "passo_pct"
# (n_bins = round(100/passo_pct) fazia vários passo_pct diferentes colapsarem
# no mesmo n_bins -- ex.: 30 e 35 -> n_bins=3; 40/45/50 -> n_bins=2).
#
# Moderada de propósito: agora que cada linha é 1 trial com as 4 áreas juntas
# (Ponto 1), o número de colunas já multiplica por ~4 -- n_bins maiores (ex. 10, 20)
# deixariam o dataset extremamente largo em relação ao número de trials (~50-70).
# Amplie com cautela (ex.: adicionar 10) se tiver tempo e quiser testar.
N_BINS_GRID = [2, 3, 4, 5, 8]
print(f"Candidatos de n_bins a testar: {N_BINS_GRID}")

# Multiclasse: as 3 condições
CONDICOES_ALVO = [25, 50, 75]

caminho_dados = next((p for p in CAMINHOS_CANDIDATOS if p.exists()), None)
if caminho_dados is None:
    raise FileNotFoundError(
        "features_all não encontrado. Ajuste CAMINHOS_CANDIDATOS para apontar "
        "para o seu features_all.parquet ou features_all.csv."
    )

if caminho_dados.suffix == ".parquet":
    df_bruto = pd.read_parquet(caminho_dados)
else:
    df_bruto = pd.read_csv(caminho_dados)

print(f" Carregado: {caminho_dados}")
print(f"   Shape: {df_bruto.shape[0]:,} linhas × {df_bruto.shape[1]} colunas")

pasta_multiclasse = Path("resultados_multiclasse_v2")
pasta_multiclasse.mkdir(exist_ok=True)
print(f"   Resultados serão salvos em: {pasta_multiclasse.resolve()}")


## 2. Limpeza dos Dados

**Correção do Ponto 8:** o filtro de NOR passa a ser feito primeiro pela coluna
`condicao` (mais confiável), com a busca por `"nor"` no nome do arquivo como camada
extra de segurança (não como critério único). No final, um `assert` confirma que só
sobraram as 3 condições esperadas.

In [ ]:
duracao_por_stem = df_bruto.groupby("stem")["segundo"].max()
stems_curtos = duracao_por_stem[duracao_por_stem < DURACAO_MINIMA_S]

n_arquivos_antes = df_bruto["stem"].nunique()

# CORREÇÃO DO PONTO 8: critério primário = condicao numérica válida (25/50/75%).
# Isso já exclui NOR (e qualquer outra coisa) sem depender do nome do arquivo.
condicao_numerica = pd.to_numeric(df_bruto["condicao"], errors="coerce")
condicoes_validas = {c / 100 for c in CONDICOES_ALVO}
mask_condicao_valida = condicao_numerica.apply(lambda v: any(np.isclose(v, c, atol=1e-6) for c in condicoes_validas))
stems_condicao_invalida = df_bruto.loc[~mask_condicao_valida, "stem"].unique()

# Camada extra de segurança (redundante com o filtro acima, mas barata):
stems_nor_por_nome = df_bruto.loc[
    df_bruto["stem"].str.contains("nor", case=False, na=False), "stem"
].unique()

stems_para_remover = set(stems_condicao_invalida) | set(stems_nor_por_nome) | set(stems_curtos.index)

df_limpo = df_bruto[~df_bruto["stem"].isin(stems_para_remover)].copy()

n_arquivos_depois = df_limpo["stem"].nunique()

print("=" * 60)
print(f"Arquivos antes das exclusões        : {n_arquivos_antes}")
print(f"Arquivos com condição inválida (NOR): {len(stems_condicao_invalida)}")
print(f"Arquivos com 'nor' no nome (extra)  : {len(stems_nor_por_nome)}")
print(f"Arquivos < {DURACAO_MINIMA_S}s removidos       : {len(stems_curtos)}")
print(f"Arquivos após as exclusões           : {n_arquivos_depois}")
print("=" * 60)

AREAS = sorted({c[:-len("_mean")] for c in df_limpo.columns if c.endswith("_mean")})
print(f"\nÁreas anatômicas encontradas ({len(AREAS)}): {AREAS}")

condicoes_restantes = sorted(pd.to_numeric(df_limpo["condicao"], errors="coerce").dropna().unique())
print(f"Condições restantes no dataset: {condicoes_restantes}  (deve conter só 0.25, 0.5 e 0.75)")
assert set(np.round(condicoes_restantes, 2)) <= {0.25, 0.5, 0.75}, (
    "Ainda há condições fora de {25%, 50%, 75%} em df_limpo -- confira a coluna 'condicao'."
)


## 3. Engenharia de Atributos (parametrizada por `n_bins`)

**Correção do Ponto 1** (a mais importante): antes, cada `(trial, área)` virava uma
linha separada -- o modelo via só uma área por vez, nunca a relação entre elas no
mesmo trial. Agora `transformar_wide_por_trial` gera **uma linha por trial** (`stem`),
com as áreas como **blocos de colunas** (`CA3b_pot_theta_mean_25pct`,
`CA3c_pot_theta_mean_25pct`, ...) -- o modelo passa a enxergar todas as áreas
simultaneamente.

Como consequência, um trial pode não ter todas as áreas registradas -- nesse caso o
bloco de colunas daquela área fica com `NaN`. Por isso o pipeline (Seção 4) ganhou um
`SimpleImputer` como primeiro passo (antes do `StandardScaler`), refeito a cada fold
como qualquer outro passo do `Pipeline` -- sem vazamento.

In [ ]:
def atribuir_faixa_percentual(n_segundos, n_bins):
    posicao_relativa = (np.arange(n_segundos) + 1) / n_segundos
    faixa = np.ceil(posicao_relativa * n_bins).astype(int)
    return np.clip(faixa, 1, n_bins)


def transformar_wide_por_trial(df_long, areas, features_por_area, n_bins):
    """UMA LINHA POR TRIAL (stem). As áreas viram blocos de colunas com prefixo
    '<area>_...' em vez de uma linha por área -- corrige o Ponto 1 da revisão
    (o modelo agora vê todas as áreas do mesmo trial simultaneamente).
    Trials sem alguma área registrada ficam com NaN no bloco daquela área
    (tratado depois por SimpleImputer dentro do Pipeline)."""
    passo_pct = 100 // n_bins
    rotulos_pct = [passo_pct * b for b in range(1, n_bins + 1)]

    df_long = df_long.sort_values(["stem", "segundo"]).copy()
    linhas_resultado = []

    for stem, grupo in df_long.groupby("stem", sort=False):
        grupo = grupo.sort_values("segundo")
        grupo = grupo.assign(_faixa_pct=atribuir_faixa_percentual(len(grupo), n_bins))

        rato = grupo["rato"].iloc[0]
        trial = grupo["trial"].iloc[0]
        condicao = grupo["condicao"].iloc[0]

        linha = {
            "stem": stem, "rato": rato, "trial": trial,
            "condicao": condicao, "duracao_s": len(grupo),
        }

        for area in areas:
            col_ref = f"{area}_mean"
            area_registrada = col_ref in grupo.columns and grupo[col_ref].notna().any()
            linha[f"{area}_registrada"] = int(area_registrada)

            for feature in features_por_area:
                if feature == FEATURE_BOOLEANA:
                    continue
                coluna = f"{area}_{feature}"
                prefixo = f"{area}_{feature}"
                tem_coluna = area_registrada and coluna in grupo.columns

                for faixa, pct in zip(range(1, n_bins + 1), rotulos_pct):
                    if not tem_coluna:
                        for estat in ESTATISTICAS_POR_JANELA:
                            linha[f"{prefixo}_{estat}_{pct}pct"] = np.nan
                        continue
                    valores = grupo.loc[grupo["_faixa_pct"] == faixa, coluna]
                    linha[f"{prefixo}_mean_{pct}pct"] = valores.mean()
                    linha[f"{prefixo}_std_{pct}pct"] = valores.std() if len(valores) > 1 else np.nan
                    linha[f"{prefixo}_min_{pct}pct"] = valores.min() if len(valores) else np.nan
                    linha[f"{prefixo}_max_{pct}pct"] = valores.max() if len(valores) else np.nan
                    linha[f"{prefixo}_median_{pct}pct"] = valores.median()

            coluna_bool = f"{area}_{FEATURE_BOOLEANA}"
            tem_bool = area_registrada and coluna_bool in grupo.columns
            for faixa, pct in zip(range(1, n_bins + 1), rotulos_pct):
                if tem_bool:
                    valores = grupo.loc[grupo["_faixa_pct"] == faixa, coluna_bool]
                    linha[f"{area}_{FEATURE_BOOLEANA}_true_rate_{pct}pct"] = (
                        (valores == True).mean() if len(valores) else np.nan
                    )
                else:
                    linha[f"{area}_{FEATURE_BOOLEANA}_true_rate_{pct}pct"] = np.nan

        linhas_resultado.append(linha)

    return pd.DataFrame(linhas_resultado)


def remover_colunas_correlacionadas(X_ref, colunas_protegidas, limiar=0.8):
    """Identifica atributos com |correlação| > limiar (colunas protegidas ficam de fora).
    corr() ignora NaN par a par -- funciona normalmente mesmo com os blocos de área
    ausente."""
    colunas_avaliar = [c for c in X_ref.columns if c not in colunas_protegidas]
    corr = X_ref[colunas_avaliar].corr().abs()
    mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
    upper = corr.where(mask)
    return [c for c in upper.columns if any(upper[c] > limiar)]


def _montar_wide_com_derivadas(df_long, n_bins):
    """Wide transform (1 linha por trial) + colunas derivadas por área
    (_media_geral, taxa geral do PAC alerta)."""
    df_pct = transformar_wide_por_trial(df_long, AREAS, FEATURES_POR_AREA, n_bins=n_bins)

    features_continuas = [f for f in FEATURES_POR_AREA if f != FEATURE_BOOLEANA]
    passo_pct_real = 100 // n_bins
    pcts = [passo_pct_real * b for b in range(1, n_bins + 1)]

    for area in AREAS:
        for feature in features_continuas:
            cols = [f"{area}_{feature}_mean_{p}pct" for p in pcts]
            df_pct[f"{area}_{feature}_media_geral"] = df_pct[cols].mean(axis=1)
        df_pct[f"{area}_pac_sl_alerta_taxa_geral"] = df_pct[
            [f"{area}_pac_sl_alerta_true_rate_{p}pct" for p in pcts]
        ].mean(axis=1)

    df_pct["condicao"] = pd.to_numeric(df_pct["condicao"])
    df_pct = df_pct.drop(columns=["trial", "duracao_s", "stem"])
    return df_pct


def montar_dataset_modelagem(df_long_limpo, n_bins, verbose=True):
    """Reproduz a engenharia de atributos completa (transformação wide por trial +
    remoção de correlacionados) para um dado `n_bins`, usando TODOS os dados
    recebidos. Usada para o "modelo final" de interpretação (SHAP) -- NÃO usada
    para reportar acurácia (isso é feito por `montar_dataset_treino_teste`,
    abaixo, dentro da CV aninhada)."""
    df_pct = _montar_wide_com_derivadas(df_long_limpo, n_bins)

    groups = df_pct["rato"]
    y = df_pct["condicao"]
    X = df_pct.drop(columns=["rato", "condicao"], errors="ignore")

    colunas_removidas = remover_colunas_correlacionadas(X, colunas_protegidas=[])
    features_finais = [c for c in X.columns if c not in colunas_removidas]
    X_final = X[features_finais]

    if verbose:
        print(f"[n_bins={n_bins}] atributos: {X.shape[1]} -> {X_final.shape[1]} "
              f"({len(colunas_removidas)} removidos por correlação)")

    return X_final, y, groups, colunas_removidas


def montar_dataset_treino_teste(df_treino_long, df_teste_long, n_bins):
    """Monta treino e teste SEPARADAMENTE para um fold da CV aninhada.

    A remoção de atributos correlacionados é decidida usando só `df_treino_long`
    -- o rato de teste (em `df_teste_long`) nunca participa dessa decisão.
    Como as áreas agora são blocos de colunas fixos (não mais categorias
    dummy), treino e teste já têm as mesmas colunas por construção."""
    df_pct_treino = _montar_wide_com_derivadas(df_treino_long, n_bins)
    df_pct_teste = _montar_wide_com_derivadas(df_teste_long, n_bins)

    groups_treino = df_pct_treino["rato"]
    y_treino = df_pct_treino["condicao"]
    X_treino = df_pct_treino.drop(columns=["rato", "condicao"], errors="ignore")

    groups_teste = df_pct_teste["rato"]
    y_teste = df_pct_teste["condicao"]
    X_teste = df_pct_teste.drop(columns=["rato", "condicao"], errors="ignore")

    colunas_removidas = remover_colunas_correlacionadas(X_treino, colunas_protegidas=[])
    features_finais = [c for c in X_treino.columns if c not in colunas_removidas]

    X_treino_final = X_treino[features_finais]
    X_teste_final = X_teste[features_finais]

    return X_treino_final, y_treino, groups_treino, X_teste_final, y_teste, groups_teste


print("Funções de engenharia de atributos definidas (1 linha por trial, CV aninhada).")


## 4. Funções Genéricas — Pipeline + CV Aninhada

Mesma lógica de CV aninhada da v1 (Pontos 2 e 3 da revisão, já corrigidos lá), com
duas mudanças:

- O `Pipeline` ganhou um passo `SimpleImputer` **antes** do `StandardScaler`, pra
  lidar com os `NaN` dos blocos de área ausente (refeito a cada fold, sem vazamento).
- A busca interna agora itera sobre `n_bins` (Ponto 4) em vez de `passo_pct`.

In [ ]:
def montar_pipeline_multiclasse(estimator, usar_selecao_features=True, selector_estimator=None):
    """Pipeline genérico: SimpleImputer -> StandardScaler -> SelectFromModel -> estimator."""
    passos = [("imputer", SimpleImputer(strategy="median"))]
    passos.append(("scaler", StandardScaler()))
    if usar_selecao_features:
        sel_est = selector_estimator or RandomForestClassifier(
            n_estimators=300, max_depth=5, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1,  # single-thread: evita paralelismo aninhado
        )
        passos.append(("selector", SelectFromModel(sel_est, threshold=-np.inf, max_features=20)))
    passos.append(("clf", estimator))
    return Pipeline(passos)


def _scorer_balanced_accuracy_silencioso(estimator, X, y):
    y_pred = estimator.predict(X)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        return balanced_accuracy_score(y, y_pred)


def escolher_configuracao_interna(df_treino_long, estimator, param_grid_semsel, param_grid_comsel,
                                   n_bins_grid, y_transformer=None, rotulo="modelo"):
    """CV INTERNA: escolhe n_bins, com/sem seleção de features e hiperparâmetros
    usando SÓ os ratos de treino deste fold externo (LOGO interno sobre eles).
    O rato de teste do fold externo nunca aparece aqui."""
    y_transformer = y_transformer or (lambda y: y)
    logo_interno = LeaveOneGroupOut()

    melhor_global = None  # (score, n_bins, usa_selecao, params)

    for n_bins in n_bins_grid:
        X_pct, y_pct, groups_pct, _ = montar_dataset_modelagem(df_treino_long, n_bins, verbose=False)
        y_pct = y_transformer((y_pct * 100).round().astype(int))

        if groups_pct.nunique() < 2:
            print(f"    [{rotulo}] n_bins={n_bins}: menos de 2 ratos de treino -- pulando.")
            continue

        cv_splits_interno = list(logo_interno.split(X_pct, y_pct, groups=groups_pct))

        pipe_semsel = montar_pipeline_multiclasse(clone(estimator), usar_selecao_features=False)
        busca_semsel = GridSearchCV(
            pipe_semsel, param_grid=param_grid_semsel, scoring=_scorer_balanced_accuracy_silencioso,
            cv=cv_splits_interno, n_jobs=-1, refit=False,
        )
        busca_semsel.fit(X_pct, y_pct)
        candidatos = [(busca_semsel.best_score_, n_bins, False, dict(busca_semsel.best_params_))]

        pipe_comsel = montar_pipeline_multiclasse(clone(estimator), usar_selecao_features=True)
        busca_comsel = GridSearchCV(
            pipe_comsel, param_grid=param_grid_comsel, scoring=_scorer_balanced_accuracy_silencioso,
            cv=cv_splits_interno, n_jobs=-1, refit=False,
        )
        busca_comsel.fit(X_pct, y_pct)
        candidatos.append((busca_comsel.best_score_, n_bins, True, dict(busca_comsel.best_params_)))

        for cand in candidatos:
            if melhor_global is None or cand[0] > melhor_global[0]:
                melhor_global = cand

    score, n_bins, usa_selecao, params = melhor_global
    print(f"    [{rotulo}] config interna escolhida: n_bins={n_bins} | "
          f"seleção={'com' if usa_selecao else 'sem'} | CV interna={score:.3f} | params={params}")
    return {"n_bins": n_bins, "usa_selecao": usa_selecao, "params": params}


def treinar_avaliar_aninhado(df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
                              n_bins_grid=None, y_transformer=None, inverse_label_fn=None):
    """Avaliação com CV ANINHADA (nested Leave-One-Rat-Out):
    - loop EXTERNO: cada rato vira teste uma vez;
    - dentro de cada fold externo, a escolha de n_bins / seleção de features /
      hiperparâmetros usa só os ratos de TREINO daquele fold (CV interna) --
      o rato de teste nunca participa dessa escolha.
    Como cada linha já é 1 trial (Ponto 1), a métrica é naturalmente por trial
    (Ponto 6) -- sem precisar agregar previsões de várias áreas depois."""
    n_bins_grid = n_bins_grid or N_BINS_GRID
    y_transformer = y_transformer or (lambda y: y)
    ratos = sorted(df_long_limpo["rato"].unique())
    labels_multi = sorted(y_transformer(pd.Series(CONDICOES_ALVO)).tolist())

    resultados_fold = []
    configs_fold = []
    y_true_all, y_pred_all = [], []

    for i, rato_teste in enumerate(ratos, start=1):
        print(f"\n{'='*70}\n[{nome_modelo}] Fold externo {i}/{len(ratos)} -- rato de teste: {rato_teste}\n{'='*70}")

        df_treino_long = df_long_limpo[df_long_limpo["rato"] != rato_teste]
        df_teste_long = df_long_limpo[df_long_limpo["rato"] == rato_teste]

        config = escolher_configuracao_interna(
            df_treino_long, estimator, param_grid_semsel, param_grid_comsel,
            n_bins_grid, y_transformer=y_transformer, rotulo=nome_modelo,
        )
        configs_fold.append({"rato_teste": rato_teste, **config})

        X_tr, y_tr, _, X_te, y_te, _ = montar_dataset_treino_teste(
            df_treino_long, df_teste_long, config["n_bins"]
        )
        y_tr = y_transformer((y_tr * 100).round().astype(int))
        y_te = y_transformer((y_te * 100).round().astype(int))

        pipe = montar_pipeline_multiclasse(clone(estimator), config["usa_selecao"])
        pipe.set_params(**config["params"])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            bal_acc = balanced_accuracy_score(y_te, y_pred)
        f1_macro = f1_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)
        precision_macro = precision_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)
        recall_macro = recall_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)

        y_true_all.extend(y_te.tolist())
        y_pred_all.extend(list(y_pred))

        resultados_fold.append({
            "fold": i, "rato_teste": rato_teste, "n_trials_teste": len(y_te),
            "n_bins": config["n_bins"], "usa_selecao_features": config["usa_selecao"],
            "balanced_accuracy": bal_acc, "f1_macro": f1_macro,
            "precision_macro": precision_macro, "recall_macro": recall_macro,
        })
        print(f"  [{nome_modelo}] fold {i}/{len(ratos)} -- teste=rato {rato_teste} (n={len(y_te)} trials) "
              f"| n_bins={config['n_bins']} | bal_acc={bal_acc:.3f} | f1_macro={f1_macro:.3f}")

    df_resultados = pd.DataFrame(resultados_fold)
    matriz_confusao = confusion_matrix(y_true_all, y_pred_all, labels=labels_multi)

    labels_exibicao = [f"{c}%" for c in labels_multi]
    if inverse_label_fn is not None:
        labels_exibicao = [f"{c}%" for c in inverse_label_fn(labels_multi)]

    return {
        "nome_modelo": nome_modelo,
        "df_resultados": df_resultados,
        "df_configs_por_fold": pd.DataFrame(configs_fold),
        "matriz_confusao": matriz_confusao,
        "labels": labels_exibicao,
        "labels_multi": labels_multi,
    }


def relatar_resultado_multiclasse(resultado, titulo, chance_nivel):
    df_resultados = resultado["df_resultados"]

    print(f"\n=== Resumo (CV aninhada, por trial) -- {titulo} ===")
    print(f"Acurácia balanceada: {df_resultados['balanced_accuracy'].mean():.3f}"
          f" ± {df_resultados['balanced_accuracy'].std():.3f}")
    print(f"F1 macro       : {df_resultados['f1_macro'].mean():.3f}"
          f" ± {df_resultados['f1_macro'].std():.3f}")
    print(f"Precision macro: {df_resultados['precision_macro'].mean():.3f}"
          f" ± {df_resultados['precision_macro'].std():.3f}")
    print(f"Recall macro   : {df_resultados['recall_macro'].mean():.3f}"
          f" ± {df_resultados['recall_macro'].std():.3f}")

    print(f"\nConfiguração escolhida em cada fold externo (é normal variar entre folds "
          f"-- a CV interna é refeita a cada um):")
    display(resultado["df_configs_por_fold"])

    cm = resultado["matriz_confusao"]
    labels_plot = resultado["labels"]

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels_plot))); ax.set_xticklabels(labels_plot)
    ax.set_yticks(range(len(labels_plot))); ax.set_yticklabels(labels_plot)
    ax.set_xlabel("Predito"); ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão (agregada, por trial) -- {titulo}")
    limiar = cm.max() / 2 if cm.max() else 1
    for i in range(len(labels_plot)):
        for j in range(len(labels_plot)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > limiar else "black")
    fig.colorbar(im, ax=ax, label="n° de trials")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(df_resultados["fold"], df_resultados["balanced_accuracy"], marker="o")
    ax.axhline(chance_nivel, color="black", linestyle=":", label=f"chance ({chance_nivel:.2f})")
    ax.set_xticks(df_resultados["fold"])
    ax.set_xticklabels(df_resultados["rato_teste"], rotation=45, ha="right")
    ax.set_xlabel("Rato de teste (fold externo)"); ax.set_ylabel("balanced_accuracy")
    ax.set_title(f"balanced_accuracy por fold externo -- {titulo}")
    ax.legend()
    plt.tight_layout()
    plt.show()


def treinar_modelo_final_interpretacao(df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
                                        n_bins_grid=None, y_transformer=None, rotulo="modelo"):
    """Treina um modelo final em TODOS os ratos, só para fins de interpretação
    (SHAP / importância de features) -- NÃO é usado para reportar acurácia.
    A escolha de configuração aqui usa CV normal (não aninhada) sobre todos os
    ratos, o que é aceitável porque nenhuma acurácia é reportada a partir dele."""
    n_bins_grid = n_bins_grid or N_BINS_GRID
    y_transformer = y_transformer or (lambda y: y)

    config = escolher_configuracao_interna(
        df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
        n_bins_grid, y_transformer=y_transformer, rotulo=f"{rotulo} [modelo final]",
    )
    X_final, y_final, groups_final, _ = montar_dataset_modelagem(
        df_long_limpo, config["n_bins"], verbose=False
    )
    y_final = y_transformer((y_final * 100).round().astype(int))

    pipe_final = montar_pipeline_multiclasse(clone(estimator), config["usa_selecao"])
    pipe_final.set_params(**config["params"])
    pipe_final.fit(X_final, y_final)

    return {"pipe_final": pipe_final, "X_final": X_final, "y_final": y_final,
            "groups_final": groups_final, "config": config}


def calcular_e_plotar_shap(pipe, X, rotulo, max_amostras=200):
    """Calcula os valores SHAP do classificador final (já treinado em todos os
    dados) e plota as top-15 features por |SHAP| médio."""
    if not SHAP_DISPONIVEL:
        print(f"[{rotulo}] shap não disponível -- pulando etapa de SHAP.")
        return None

    X_proc = X
    for _, passo in pipe.steps[:-1]:
        X_proc = passo.transform(X_proc)
    selector = pipe.named_steps.get("selector")
    feats = X.columns[selector.get_support()] if selector is not None else X.columns
    X_proc = pd.DataFrame(np.asarray(X_proc), columns=feats, index=X.index)

    n_amostra = min(max_amostras, len(X_proc))
    X_amostra = X_proc.sample(n_amostra, random_state=RANDOM_STATE) if len(X_proc) > n_amostra else X_proc

    clf = pipe.named_steps["clf"]
    # Modelos de árvore (RF, XGBoost) usam TreeExplainer explicitamente com
    # feature_perturbation="tree_path_dependent" -- o shap.Explainer genérico
    # pode falhar em versões recentes do XGBoost com tree_method="hist".
    modelos_arvore = (RandomForestClassifier,) + ((XGBClassifier,) if XGBOOST_DISPONIVEL else ())
    if isinstance(clf, modelos_arvore):
        explainer = shap.TreeExplainer(clf, feature_perturbation="tree_path_dependent")
    else:
        explainer = shap.Explainer(clf, X_proc)
    valores_shap = explainer(X_amostra)

    valores = valores_shap.values
    valores_abs_medios = (
        np.abs(valores).mean(axis=(0, 2)) if valores.ndim == 3 else np.abs(valores).mean(axis=0)
    )
    resumo_shap = pd.Series(valores_abs_medios, index=feats).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    top_shap = resumo_shap.head(15).iloc[::-1]
    ax.barh(top_shap.index, top_shap.values, color="#55A868")
    ax.set_xlabel("|SHAP| médio")
    ax.set_title(f"Top 15 features (SHAP) -- {rotulo}")
    plt.tight_layout()
    plt.show()

    return resumo_shap


def pipeline_multiclasse_aninhado(df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
                                   n_bins_grid=None, y_transformer=None, inverse_label_fn=None,
                                   calcular_shap_flag=True, shap_max_amostras=200):
    n_bins_grid = n_bins_grid or N_BINS_GRID
    chance_nivel = 1 / len(CONDICOES_ALVO)

    print(f"\n{'#'*70}\n# {nome_modelo} -- CV ANINHADA (nested Leave-One-Rat-Out)\n{'#'*70}")
    resultado = treinar_avaliar_aninhado(
        df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
        n_bins_grid=n_bins_grid, y_transformer=y_transformer, inverse_label_fn=inverse_label_fn,
    )
    relatar_resultado_multiclasse(resultado, titulo=nome_modelo, chance_nivel=chance_nivel)

    print(f"\n{'#'*70}\n# {nome_modelo} -- modelo final (todos os ratos) p/ interpretação (SHAP)\n{'#'*70}")
    modelo_final = treinar_modelo_final_interpretacao(
        df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
        n_bins_grid=n_bins_grid, y_transformer=y_transformer, rotulo=nome_modelo,
    )

    resumo_shap = None
    if calcular_shap_flag:
        resumo_shap = calcular_e_plotar_shap(
            modelo_final["pipe_final"], modelo_final["X_final"], rotulo=nome_modelo, max_amostras=shap_max_amostras,
        )

    return {
        "nome_modelo": nome_modelo,
        "resultado": resultado,
        "modelo_final": modelo_final,
        "resumo_shap": resumo_shap,
    }


print("Funções genéricas (pipeline + CV aninhada, multiclasse por trial) definidas.")


## 5. RandomForest — Pipeline Completo (CV aninhada)

In [ ]:
PARAM_GRID_RF_SEMSEL = {
    "clf__n_estimators": [100, 200, 400],
    "clf__max_depth": [3, 5, 8, None],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__max_features": ["sqrt", "log2", 0.5],
    "clf__class_weight": ["balanced"],
}
PARAM_GRID_RF_COMSEL = {
    "selector__max_features": TOP_N_FEATURES_GRID,
    **PARAM_GRID_RF_SEMSEL,
}

rf_base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)

saida_rf = pipeline_multiclasse_aninhado(
    df_limpo, nome_modelo="RandomForest (25/50/75%)", estimator=rf_base,
    param_grid_semsel=PARAM_GRID_RF_SEMSEL, param_grid_comsel=PARAM_GRID_RF_COMSEL,
)


## 6. Regressão Logística — Pipeline Completo (CV aninhada)

In [ ]:
PARAM_GRID_LOGREG_SEMSEL = {
    "clf__C": [1e-3, 1e-2, 1e-1, 1, 10, 100],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"],
    "clf__class_weight": [None, "balanced"],
    "clf__max_iter": [5000],
}
PARAM_GRID_LOGREG_COMSEL = {
    "selector__max_features": TOP_N_FEATURES_GRID,
    **PARAM_GRID_LOGREG_SEMSEL,
}

logreg_base = LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE)

saida_logreg = pipeline_multiclasse_aninhado(
    df_limpo, nome_modelo="Regressão Logística (25/50/75%)", estimator=logreg_base,
    param_grid_semsel=PARAM_GRID_LOGREG_SEMSEL, param_grid_comsel=PARAM_GRID_LOGREG_COMSEL,
)


## 7. XGBoost — Pipeline Completo (CV aninhada)

De volta a multiclasse (`objective="multi:softprob"`, `num_class=3`) -- os rótulos
`25/50/75` são recodificados para `0/1/2` via `y_transformer`, e os relatórios finais
são traduzidos de volta via `inverse_label_fn`.

In [ ]:
if XGBOOST_DISPONIVEL:
    MAPA_XGB = {c: i for i, c in enumerate(CONDICOES_ALVO)}   # {25: 0, 50: 1, 75: 2}
    MAPA_XGB_INV = {i: c for c, i in MAPA_XGB.items()}

    def codificar_y_xgb(y_series):
        return y_series.map(MAPA_XGB)

    def decodificar_y_xgb(labels_codificados):
        return [MAPA_XGB_INV[l] for l in labels_codificados]

    PARAM_GRID_XGB_SEMSEL = {
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [2, 3, 5],
        "clf__learning_rate": [0.03, 0.1, 0.2],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__min_child_weight": [1, 5],
    }
    PARAM_GRID_XGB_COMSEL = {
        "selector__max_features": TOP_N_FEATURES_GRID,
        **PARAM_GRID_XGB_SEMSEL,
    }

    xgb_base = XGBClassifier(
        objective="multi:softprob", num_class=len(CONDICOES_ALVO), eval_metric="mlogloss",
        random_state=RANDOM_STATE, n_jobs=1, tree_method="hist",
    )

    saida_xgb = pipeline_multiclasse_aninhado(
        df_limpo, nome_modelo="XGBoost (25/50/75%)", estimator=xgb_base,
        param_grid_semsel=PARAM_GRID_XGB_SEMSEL, param_grid_comsel=PARAM_GRID_XGB_COMSEL,
        y_transformer=codificar_y_xgb, inverse_label_fn=decodificar_y_xgb,
    )
else:
    saida_xgb = None
    print("Pulando seção XGBoost -- pacote 'xgboost' não disponível (pip install xgboost).")


## 8. Teste de Permutação (config fixa por fold, simplificação documentada)

Mesma simplificação pragmática da v1: refazer a CV interna para cada permutação
multiplicaria o custo já alto por `n_permutacoes` -- inviável no prazo. Reaproveitamos
a configuração já escolhida em cada fold externo e só re-treinamos/reavaliamos com os
rótulos embaralhados **por trial** (`stem`). Rodamos só para o modelo com maior
acurácia observada.

**Limitação a admitir se perguntarem:** a distribuição nula não captura a variância de
"escolher a configuração vencedora", então o p-valor é uma aproximação prática.

In [ ]:
def teste_permutacao_aninhado_simplificado(df_long_limpo, resultado_aninhado, estimator,
                                            y_transformer=None, n_permutacoes=50,
                                            random_state=RANDOM_STATE, rotulo="modelo"):
    rng = np.random.RandomState(random_state)
    y_transformer = y_transformer or (lambda y: y)
    df_configs = resultado_aninhado["df_configs_por_fold"].set_index("rato_teste")
    bal_acc_observado = resultado_aninhado["df_resultados"]["balanced_accuracy"].mean()
    ratos = resultado_aninhado["df_resultados"]["rato_teste"].tolist()

    bal_acc_nulo = []
    for p in range(n_permutacoes):
        mapa_stem = df_long_limpo.drop_duplicates("stem")[["stem", "condicao"]].copy()
        mapa_stem["condicao_perm"] = rng.permutation(mapa_stem["condicao"].values)
        df_perm = df_long_limpo.merge(mapa_stem[["stem", "condicao_perm"]], on="stem", how="left")
        df_perm = df_perm.drop(columns=["condicao"]).rename(columns={"condicao_perm": "condicao"})

        bal_accs_fold = []
        for rato_teste in ratos:
            cfg = df_configs.loc[rato_teste]
            df_treino_long = df_perm[df_perm["rato"] != rato_teste]
            df_teste_long = df_perm[df_perm["rato"] == rato_teste]
            X_tr, y_tr, _, X_te, y_te, _ = montar_dataset_treino_teste(
                df_treino_long, df_teste_long, cfg["n_bins"]
            )
            y_tr = y_transformer((y_tr * 100).round().astype(int))
            y_te = y_transformer((y_te * 100).round().astype(int))
            pipe = montar_pipeline_multiclasse(clone(estimator), cfg["usa_selecao"])
            pipe.set_params(**cfg["params"])
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                bal_accs_fold.append(balanced_accuracy_score(y_te, y_pred))
        bal_acc_nulo.append(np.mean(bal_accs_fold))
        if (p + 1) % 10 == 0:
            print(f"  [{rotulo}] permutação {p+1}/{n_permutacoes}...")

    bal_acc_nulo = np.array(bal_acc_nulo)
    p_valor = (np.sum(bal_acc_nulo >= bal_acc_observado) + 1) / (n_permutacoes + 1)

    print(f"\n[{rotulo}] balanced_accuracy observado: {bal_acc_observado:.3f}")
    print(f"[{rotulo}] balanced_accuracy nulo (média ± dp): {bal_acc_nulo.mean():.3f} ± {bal_acc_nulo.std():.3f}")
    print(f"[{rotulo}] p-valor (permutação, n={n_permutacoes}, config fixa por fold): {p_valor:.4f}")

    chance_nivel = 1 / len(CONDICOES_ALVO)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(bal_acc_nulo, bins=20, color="#999999", alpha=0.8, label="distribuição nula (rótulos embaralhados por trial)")
    ax.axvline(bal_acc_observado, color="crimson", linewidth=2, label=f"observado = {bal_acc_observado:.3f}")
    ax.axvline(chance_nivel, color="black", linestyle=":", label=f"chance ({chance_nivel:.2f})")
    ax.set_xlabel("balanced_accuracy (média nested-LOGO)")
    ax.set_ylabel("frequência")
    ax.set_title(f"Teste de permutação (config fixa por fold) -- {rotulo} (n={n_permutacoes}) -- p={p_valor:.4f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return {"bal_acc_observado": bal_acc_observado, "bal_acc_nulo": bal_acc_nulo, "p_valor": p_valor}


saidas_disponiveis = [s for s in [saida_rf, saida_logreg, saida_xgb] if s is not None]
melhor_saida = max(
    saidas_disponiveis,
    key=lambda s: s["resultado"]["df_resultados"]["balanced_accuracy"].mean(),
)
print(f"Modelo com maior balanced_accuracy observada: {melhor_saida['nome_modelo']}")

usar_xgb = XGBOOST_DISPONIVEL and (melhor_saida is saida_xgb)
y_transformer_melhor = codificar_y_xgb if usar_xgb else (lambda y: y)
estimator_melhor = xgb_base if usar_xgb else (
    rf_base if melhor_saida is saida_rf else logreg_base
)

resultado_permutacao = teste_permutacao_aninhado_simplificado(
    df_limpo, melhor_saida["resultado"], clone(estimator_melhor),
    y_transformer=y_transformer_melhor, n_permutacoes=50, rotulo=melhor_saida["nome_modelo"],
)


## 9. Comparação Final e Exportação dos Resultados

In [ ]:
linhas_comparacao = []
for saida in saidas_disponiveis:
    df_r = saida["resultado"]["df_resultados"]
    linhas_comparacao.append({
        "modelo": saida["nome_modelo"],
        "balanced_accuracy_media": df_r["balanced_accuracy"].mean(),
        "balanced_accuracy_dp": df_r["balanced_accuracy"].std(),
        "f1_macro_media": df_r["f1_macro"].mean(),
        "precision_macro_media": df_r["precision_macro"].mean(),
        "recall_macro_media": df_r["recall_macro"].mean(),
        "n_folds": len(df_r),
    })

df_comparacao = pd.DataFrame(linhas_comparacao).sort_values("balanced_accuracy_media", ascending=False)
print("\n=== Comparação final -- RF vs LogReg vs XGBoost (25/50/75%, CV aninhada, por trial) ===")
display(df_comparacao.round(3))

chance_nivel = 1 / len(CONDICOES_ALVO)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_comparacao["modelo"], df_comparacao["balanced_accuracy_media"],
       yerr=df_comparacao["balanced_accuracy_dp"], color="#4C72B0")
ax.axhline(chance_nivel, color="black", linestyle=":", label=f"chance ({chance_nivel:.2f})")
ax.set_ylabel("balanced_accuracy (CV aninhada, por trial)")
ax.set_title("Comparação final -- 25% vs 50% vs 75%")
ax.legend()
plt.tight_layout()
plt.show()

df_comparacao.to_csv(pasta_multiclasse / "comparacao_final_v2.csv", index=False)

for saida in saidas_disponiveis:
    nome_arquivo = (
        saida["nome_modelo"].lower()
        .replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct").replace("/", "_")
    )
    saida["resultado"]["df_resultados"].to_csv(
        pasta_multiclasse / f"resultados_por_fold_{nome_arquivo}.csv", index=False
    )
    saida["resultado"]["df_configs_por_fold"].to_csv(
        pasta_multiclasse / f"configs_por_fold_{nome_arquivo}.csv", index=False
    )

print(f"\n✅ Resultados salvos em: {pasta_multiclasse.resolve()}")
